# Master catalog of games

Builds one row per game (Whizdom Game ID) from **ref_games**, joined with the other sources.
The functions are in `master_lib.py` (tests: `tests/test_master_lib.py`).

**Keys**
- Game: name + provider. The provider is the studio that made the game, not the platform.
- `whizdom_game_id = uuid5(NAMESPACE, "<provider>/<game>")`: the same game always gets the same ID.
- `provider_id = uuid5(NAMESPACE, "<provider>")`.

**Cell types**
- **MANUAL**: reads ClickHouse or S3. Only run them by hand. Their flags are off by default.
- Everything else works with local files.

**Output**: `output/master/master_catalog.xlsx`.

In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

HERE = Path.cwd() if (Path.cwd() / "master_lib.py").exists() else Path.cwd() / "catalog_project"
sys.path[:0] = [str(HERE), str(HERE.parent)]
import master_lib as ml  # noqa: E402
import slotcatalog_json  # noqa: E402
from priority_list import by_key, key  # noqa: E402

config = yaml.safe_load((HERE / "config.yaml").read_text())
OUT = HERE / config["master_out"]
OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 200)
print("ID version", ml.ID_VERSION, "| namespace", ml.NAMESPACE)

## 1. Base list (ref_games)

- Every game and every column of ref_games (no filter by date).
- Run the MANUAL cell once to export `output/master/ref_games.parquet`.

In [2]:
# MANUAL (ClickHouse, read only): set to True to export a fresh ref_games
REFRESH_REF_GAMES = False
if REFRESH_REF_GAMES:
    from active_games import query
    from signal_audit.data.clickhouse_mirror import client
    query(client(), "ref_games_master").to_parquet(OUT / "ref_games.parquet", index=False)

In [ ]:
path = OUT / "ref_games.parquet"
assert path.exists(), "ref_games.parquet is missing: run the MANUAL cell above once (REFRESH_REF_GAMES = True)"
ref = pd.read_parquet(path)
assert ref["game_uid"].is_unique, "ref_games must have one row per game_uid"
REF_DATE = pd.Timestamp(path.stat().st_mtime, unit="s").date()

games = ref.add_prefix("ref_").reset_index(drop=True)          # the base list is the whole of ref_games
print(f"ref_games of {REF_DATE}: {len(games)} games")

## 2. Other sources

- Client Area, Slot Catalog URL file, games_enriched and the raw Slot Catalog JSON.
- The MANUAL cell downloads the JSON from S3 again, only if needed.

In [4]:
# MANUAL (S3, profile in config.yaml): set to True to download and unpack the raw JSON again
REFRESH_RAW = False
if REFRESH_RAW:
    import io, zipfile, boto3
    bucket, zip_key = config["sa_raw_zip"].removeprefix("s3://").split("/", 1)
    body = boto3.Session(profile_name=config["sa_raw_profile"]).client("s3").get_object(Bucket=bucket, Key=zip_key)["Body"].read()
    target = HERE / config["slotcatalog_json"]
    target.mkdir(exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(body)) as z:
        for n in z.namelist():
            if n.startswith("data/raw/") and n.endswith(".json"):
                (target / Path(n).name).write_bytes(z.read(n))

In [5]:
ca = pd.read_csv(HERE / config["client_area"], dtype={"Core Id Desktop": str, "Core Id Mobile": str})
sa = pd.read_excel(HERE / config["sa_urls"], dtype=str).sort_values("sc_url", na_position="last")
# the index and the ranking of the old priority file of SA are not data of the game
enriched = pd.read_excel(HERE / config["games_enriched"], dtype=str).drop(columns=["Unnamed: 0", "dervied_rank", "total_bets_pct", "totatl_bets_cumsum"])
info = slotcatalog_json.load(HERE / config["slotcatalog_json"])
raw = ml.load_raw(HERE / config["slotcatalog_json"])
print(f"Client Area {len(ca)} rows | Slot Catalog file {len(sa)} rows | games_enriched {len(enriched)} rows | raw JSON {len(raw)} files")

Client Area 16273 rows | Slot Catalog file 15855 rows | games_enriched 1071 rows | raw JSON 9532 files


## 3. Join the sources

- Client Area, Slot Catalog URL file (`scu_`) and games_enriched join by game id and platform.
- The JSON joins by the UUID of the Slot Catalog URL file.
- Studio: ref_games, then Client Area, then JSON, then platform (`studio_source`).

In [ ]:
games["key"] = key(games["ref_game_id"], games["ref_provider_code"])
ca_all = by_key(ca, ["Core Id Desktop", "Core Id Mobile"], "Platform", list(ca.columns)).rename(columns=lambda c: "ca_" + ml.snake(c))
sa_all = by_key(sa, ["game_src_id", "core_id_desktop"], "platform", list(sa.columns)).add_prefix("scu_")
ge_all = by_key(enriched, ["game_src_id", "core_desktop_id", "core_mobile_id"], "platform_code", list(enriched.columns)).add_prefix("ge_")
games = games.join(ca_all, on="key").join(sa_all, on="key").join(ge_all, on="key")

games["json_studio"] = games["scu_game_id"].map(info.set_index("sc_uuid")["sc_provider"])
games["studio"], games["studio_source"] = ml.first_labelled(
    games, {"ref_studios": "ref_games", "ca_provider": "client_area", "json_studio": "slotcatalog_json", "ref_provider_name": "platform"})
games["studio_source"].value_counts()

## 4. Provider names

- `providers.yaml` maps each spelling to one standard provider.
- `similar_provider = true`: another provider has almost the same name, not merged yet. A person decides.

In [7]:
aliases = ml.load_providers()
all_names = pd.concat([ca["Provider"], sa["provider"], info["sc_provider"], games["studio"]])
candidates = ml.provider_candidates(all_names, aliases)      # look-alike names not merged yet

games["provider_slug"], _ = ml.canonical_provider(games["studio"], aliases)
candidates

,slug,rows,core,standard
0,high5,243,high5,high5
1,high-5-games,84,high5,high-5-games


## 5. Name, studio and IDs

**Name**: ref_games first, then Client Area, Slot Catalog URL file or games_enriched (`name_source`).
- `fix_possessive` restores apostrophes (`Santa S Gifts` becomes `Santa's Gifts`).
- Sports betting has no name in any source: its name comes from `explicit_names` in `config.yaml`.
- No name at all: no ID, the game goes to the sheet `no name`.

**Studio**: the platform-only studio is replaced by the studio of the same game name in a better source, only if the RTP matches (`same_name`).

In [ ]:
games["game_name"], games["name_source"] = ml.first_labelled(
    games, {"ref_game_name": "ref_games", "ca_game_name": "client_area", "scu_game_name": "slotcatalog_file", "ge_game_name": "games_enriched"})

explicit = (games["ref_provider_code"] + "/" + games["ref_game_id"].astype(str)).map(config["explicit_names"])
use_explicit = games["game_name"].isna() & explicit.notna()          # only where no source has a name
games.loc[use_explicit, "game_name"], games.loc[use_explicit, "name_source"] = explicit[use_explicit], "explicit"

games["game_name"] = games["game_name"].map(ml.fix_possessive)
games["game_slug"] = games["game_name"].map(ml.slugify)
games["name_source"].value_counts()

In [9]:
games["rtp"], _ = ml.first_of(games, ["ref_rtp", "ca_rtp"])
games["rtp"] = pd.to_numeric(games["rtp"], errors="coerce")

same_name = ml.studio_from_same_name(games)
use = same_name["provider_slug"].notna()
games.loc[use, ["provider_slug", "studio"]] = same_name.loc[use, ["provider_slug", "studio"]]
games.loc[use, "studio_source"] = "same_name"
print(use.sum(), "games take the studio of the same game name")

167 games take the studio of the same game name


In [ ]:
# one name and provider can be two games. The Client Area name tells them apart first (Lady Wolf Moon / ... Megaways),
# then the RTP of the casino (Great Blue Jackpot 95.02 / 94.03) or the game type (Roulette / Slots)
ca_name = ml.name_from_client_area(games).map(ml.fix_possessive)
games.loc[ca_name.notna(), "game_name"], games.loc[ca_name.notna(), "name_source"] = ca_name[ca_name.notna()], "client_area"
games["game_slug"] = games["game_name"].map(ml.slugify)
base_id = [ml.whizdom_id(g, p) for g, p in zip(games["game_slug"], games["provider_slug"])]
games["variant"] = ml.variant_of(games.assign(whizdom_game_id=base_id, game_type=games["ref_game_type"].replace("", np.nan).fillna(games["ca_game_type"])))
games["whizdom_game_id"] = [ml.whizdom_id(g, p, v) for g, p, v in zip(games["game_slug"], games["provider_slug"], games["variant"])]
print(int(ca_name.notna().sum()), "game_uid take the Client Area name |", int(games["variant"].ne("").sum()), "game_uid are a variant:",
      games.loc[games["variant"].ne(""), ["game_name", "variant"]].drop_duplicates().values.tolist())
games["provider_id"] = games["provider_slug"].map(ml.provider_id)
print(games["whizdom_game_id"].notna().sum(), "games with an ID,", games["whizdom_game_id"].nunique(), "distinct IDs |",
      games["whizdom_game_id"].isna().sum(), "with no name, so no ID |", games["provider_id"].nunique(), "providers")

## 6. Match with the raw JSON

- `exact`: the JSON found by platform ids has the same name and provider.
- `review`: only one way finds the JSON, they disagree, or a close name exists (similarity 0.9 or more).
- `no_match`: nothing found.

`sc_` columns come from the JSON found by ids, or else by name and provider.

In [ ]:
js = info[["sc_uuid", "sc_provider"]].copy()
js["name"] = js["sc_uuid"].map(sa.drop_duplicates("game_id").set_index("game_id")["game_name"])
js["provider_slug"], _ = ml.canonical_provider(js["sc_provider"], aliases)
js["whizdom_game_id"] = [ml.whizdom_id(ml.slugify(n), p) for n, p in zip(js["name"], js["provider_slug"])]
js["game_slug"] = js["name"].map(ml.slugify)
js_with_id = js.dropna(subset=["whizdom_game_id"])

games["json_by_name"] = games["whizdom_game_id"].map(js_with_id.drop_duplicates("whizdom_game_id").set_index("whizdom_game_id")["sc_uuid"])
games["json_by_id"] = games["scu_game_id"].where(games["scu_game_id"].isin(set(js["sc_uuid"])))
games["json_by_id_whizdom"] = games["json_by_id"].map(js.set_index("sc_uuid")["whizdom_game_id"])

games["close_name"] = None
no_hit = games["json_by_name"].isna() & games["json_by_id"].isna()
games.loc[no_hit, "close_name"] = ml.close_names(games[no_hit], js_with_id)

games["match_status"] = ml.match_status(games["whizdom_game_id"], games["json_by_name"], games["json_by_id_whizdom"], games["close_name"])
games["sc_uuid"] = games["json_by_id"].fillna(games["json_by_name"])
games = games.join(info.set_index("sc_uuid"), on="sc_uuid")
games["match_status"].value_counts()

## 7. URL, status and flags

- `url`: Slot Catalog URL file, then JSON, then Client Area.
- `status` and `game_type`: ref_games, then Client Area.
- `same_platform_conflict`: two `game_uid` on the same platform share the ID (same game, same RTP).

In [ ]:
ca_url = games["ca_external_resource"].where(games["ca_external_resource"].astype(str).str.contains("slotcatalog"))
games["json_url"] = games["sc_uuid"].map(raw.set_index("source_id")["source_url"])
games["url"], games["url_source"] = ml.first_labelled(
    games.assign(ca_url=ca_url), {"scu_sc_url": "slotcatalog_file", "json_url": "raw_json", "ca_url": "client_area"})
games["no_url"] = games["url"].isna()

games["game_type"] = games["ref_game_type"].replace("", np.nan).fillna(games["ca_game_type"])     # ref_games writes "" for no type
games["status"], _ = ml.first_of(games, ["ref_status", "ca_status"])
games["is_sports"] = games["ref_provider_code"].isin(config["sports_provider_codes"])
games["similar_provider"] = games["provider_slug"].isin(set(candidates["standard"]))

missing = pd.DataFrame({"name": games["game_name"].isna(), "studio": games["studio_source"].isin(["platform", "none"]),
                        "game_type": games["game_type"].isna()})
games["missing_data"] = missing.apply(lambda row: ", ".join(row.index[row]), axis=1)
games["id_shared"] = games.groupby("whizdom_game_id")["ref_game_uid"].transform("count").gt(1)
games["same_platform_conflict"] = games["whizdom_game_id"].notna() & games.duplicated(["whizdom_game_id", "ref_provider_code"], keep=False)

## 8. Catalog

One row per `game_uid` of ref_games, the same list as ClickHouse and the same rows as the Games Catalog Tracker.
A game on several platforms has one Whizdom Game ID in several rows. Order of rows, the same as the 99 % list:
GGR (highest first), rounds, then `game_uid`. The first rows are the 99 % list; the rows with no GGR go after.
At the end, the games that are only in other sources (`only_in`).

Before writing, the IDs are compared with the last run. A changed ID means a rule changed.


In [13]:
identities = games[["ref_game_uid", "ref_game_id", "ref_provider_code", "ref_provider_name", "game_name", "name_source", "whizdom_game_id",
                    "provider_id", "studio_source", "match_status"]].rename(columns=lambda c: c.removeprefix("ref_"))
changes = ml.compare_ids(identities, OUT / "game_identities.parquet")
changes["change"].value_counts().to_dict() or "no changes against the last run"

{'changed': 6}

In [ ]:
activity = pd.read_csv(HERE / config["master_ggr"], usecols=["game_uid", "ggr_eur_6m", "rounds_6m"], dtype={"game_uid": str})
ACTIVITY_DATE = pd.Timestamp((HERE / config["master_ggr"]).stat().st_mtime, unit="s").date()
games = games.join(activity.set_index("game_uid"), on="ref_game_uid")

# one row per game_uid of ref_games, in the order of the 99% list; the index is the game_uid (it is not a column of the catalog)
by_id = games.dropna(subset=["whizdom_game_id"]).groupby("whizdom_game_id")
master = games.copy()
master["provider"] = master["provider_slug"]
master["id_version"] = ml.ID_VERSION
master["platforms"] = master["whizdom_game_id"].map(by_id["ref_provider_code"].agg(lambda s: ", ".join(sorted(set(s.dropna())))))
master["platform_rows"] = master["whizdom_game_id"].map(by_id.size())
master["only_in"] = ""
master = master.sort_values(["ggr_eur_6m", "rounds_6m", "ref_game_uid"], ascending=[False, False, True], na_position="last")
master.index = master["ref_game_uid"].to_numpy()
with_ggr = master["ggr_eur_6m"].notna()
assert master.index.is_unique and len(master) == len(ref), "one row per game_uid of ref_games"
assert with_ggr.iloc[:with_ggr.sum()].all() and master["ggr_eur_6m"].dropna().is_monotonic_decreasing, "rows with GGR must come first, by GGR"
print(f"{len(master)} rows (game_uid), {master['whizdom_game_id'].nunique()} IDs, {with_ggr.sum()} rows with GGR: "
      f"{master['ggr_eur_6m'].sum():,.0f} of {activity['ggr_eur_6m'].sum():,.0f} EUR of the list")


In [ ]:
# the games that are not in ref_games but are in another source, only the new ones (a new ID, or a name not in the catalog when no
# source has a provider), with the columns of their source. Their platform key is not in ref_games, or their JSON is not linked.
sc = info.set_index("sc_uuid")
keys = set(games["key"])
ca_x, ge_x, scu_x = (t[~t.index.isin(keys)] for t in (ca_all, ge_all, sa_all))
js_x = info[~info["sc_uuid"].isin(set(games["sc_uuid"].dropna()))]
ca_url = ca_x["ca_external_resource"].where(ca_x["ca_external_resource"].astype(str).str.contains("slotcatalog"))
others = pd.concat([       # in order of preference, as in the catalog: the first source with a value wins
    ca_x.assign(only_in="Client Area", name_source="client_area", game_name=ca_x["ca_game_name"],
                studio=ca_x["ca_studios"].fillna(ca_x["ca_provider"]), url=ca_url, sc_uuid=None),
    ge_x.assign(only_in="games_enriched", name_source="games_enriched", game_name=ge_x["ge_game_name"],
                studio=ge_x["ge_studio_name"].fillna(ge_x["ge_provider_name"]), url=None, sc_uuid=None),
    scu_x.assign(only_in="Slot Catalog URL file", name_source="slotcatalog_file", game_name=scu_x["scu_game_name"],
                 studio=scu_x["scu_provider"].fillna(scu_x["scu_game_id"].map(sc["sc_provider"])), url=scu_x["scu_sc_url"],
                 sc_uuid=scu_x["scu_game_id"].where(scu_x["scu_game_id"].isin(sc.index))),
    js_x.assign(only_in="Slot Catalog JSON", name_source="raw_json", game_name=js_x["sc_uuid"].map(js.set_index("sc_uuid")["name"]),
                studio=js_x["sc_provider"], url=js_x["sc_uuid"].map(raw.set_index("source_id")["source_url"])),
], ignore_index=True).dropna(subset=["game_name"])
others["game_name"] = others["game_name"].map(ml.fix_possessive)
others["provider_slug"], _ = ml.canonical_provider(others["studio"], aliases)
others["game_slug"] = others["game_name"].map(ml.slugify)
others["whizdom_game_id"] = [ml.whizdom_id(s, p) for s, p in zip(others["game_slug"], others["provider_slug"])]
new = others["whizdom_game_id"].notna() & ~others["whizdom_game_id"].isin(set(games["whizdom_game_id"].dropna()) | set(base_id))   # a variant is known too
no_provider = others["whizdom_game_id"].isna() & ~others["game_slug"].isin(set(games["game_slug"].dropna()))
others = others[new | no_provider]
group = others["whizdom_game_id"].fillna("name:" + others["game_slug"])
only_in = others.groupby(group, sort=False)["only_in"].agg(lambda s: " + ".join(dict.fromkeys(s)))
others = others.groupby(group, sort=False).first()          # one row per game: the first value of each column, in the order above
others["only_in"] = only_in
for column in sc.columns:                                    # the Slot Catalog JSON of a game of the Slot Catalog URL file
    others[column] = (others[column] if column in others else pd.Series(None, index=others.index, dtype=object)).fillna(others["sc_uuid"].map(sc[column]))
type_columns = [c for c in ["ca_game_type", "ge_game_type", "scu_game_type", "sc_game_type"] if c in others]
others = others.assign(
    provider=others["provider_slug"], provider_id=others["provider_slug"].map(lambda p: ml.provider_id(p) if isinstance(p, str) else None),
    id_version=ml.ID_VERSION, variant="", studio_source=others["name_source"], status=others.get("ca_status"),
    game_type=others[type_columns].bfill(axis=1).iloc[:, 0], is_sports=False, match_status="no_match", close_name=None,
    url_source=np.where(others["url"].notna(), others["name_source"], "none"), no_url=others["url"].isna(),
    similar_provider=others["provider_slug"].isin(set(candidates["standard"])), id_shared=False, same_platform_conflict=False,
    platforms="", platform_rows=0)
others["missing_data"] = np.where(others["studio"].isna(), "studio", "") + np.where(others["game_type"].isna(), np.where(others["studio"].isna(), ", game_type", "game_type"), "")
others.index = "only in other sources: " + others.index
master = pd.concat([master, others])
print(f"{len(others)} games only in other sources: {int(others['whizdom_game_id'].notna().sum())} new IDs, "
      f"{int(others['whizdom_game_id'].isna().sum())} names with no provider | {others['only_in'].value_counts().to_dict()} | catalog {len(master)} rows")


In [ ]:
OWN = ["whizdom_game_id", "game_name", "ggr_eur_6m", "rounds_6m", "name_source", "provider", "provider_id", "id_version", "variant", "studio",
       "studio_source", "status", "game_type", "is_sports", "rtp", "url", "url_source", "match_status", "close_name", "missing_data",
       "no_url", "similar_provider", "id_shared", "same_platform_conflict", "platforms", "platform_rows", "only_in"]
REPEATED = ["ref_game_name", "ref_status", "ref_game_type", "ca_game_type", "ref_provider_name"]   # same as game_name, status...
source_columns = [c for c in games if c.startswith(tuple(ml.SOURCES))]
left_out = ml.platform_id_columns(source_columns) + REPEATED

master = master[OWN + [c for c in source_columns if c not in left_out]]
empty = [c for c in master if c not in OWN and master[c].isna().all()]
master = master.drop(columns=empty)
print(master.shape[1], "columns | left out: ids of platforms", ml.platform_id_columns(source_columns), "| repeated", REPEATED, "| empty", empty)

## 9. games matched, games_enriched and coverage


In [ ]:
NAV = ["game_uid", "batch", "overall_status", "match_basis", "unmatched_reason", "verify_verdict", "sc_url", "all_links", "n_attribute_conflicts"]
matched = pd.concat([pd.read_excel(HERE / config["navid_final"], sheet_name=s, dtype=object, usecols=NAV) for s in ("matched", "fuzzy_review", "unmatched")], ignore_index=True)
matched = matched.rename(columns={"overall_status": "status"}).rename(columns=lambda c: c if c == "game_uid" else "gm_" + c)
BATCH = {name: f"batch {i}" for i, name in enumerate(sorted(matched["gm_batch"].dropna().unique()), 1)}   # the Excel has no names of people
matched["gm_batch"] = matched["gm_batch"].map(BATCH)

gm_fields = yaml.safe_load((HERE / "check_games_matched_final" / "fields.yaml").read_text())["fields"]
GM_FIELDS = [n for n, s in gm_fields.items() if s.get("unify", True) and not s.get("third_party_text")]
GM_FIELDS += ["description", "review_text", "pros", "cons"]          # texts of third party websites: the team wants them in the catalog
EXTRA = ["max_win_amount", "max_win_currency", "rtp_from_text", "max_win_from_text", "volatility_from_text", "rtp_differs_from_name"]
unified = pd.read_parquet(HERE / config["navid_unified"])
unified = unified[["game_uid"] + [c for n in GM_FIELDS for c in (n, f"{n}_source")] + EXTRA].rename(columns=lambda c: c if c == "game_uid" else "gm_" + c)

matched = matched.merge(unified, on="game_uid", how="left")
print(len(matched), "games in games matched |", len(unified), "unified | batches", len(BATCH))


In [ ]:
# games matched by game_uid: each game_uid takes its own row of the file; a game_uid that is not in the file takes the row of its game
# (the best status of the ID), because the attributes are of the game on any platform
gm_columns = [c for c in matched if c != "game_uid"]
best = matched.assign(wid=matched["game_uid"].map(games.set_index("ref_game_uid")["whizdom_game_id"]),
                      order=matched["gm_status"].map({"matched": 0, "fuzzy": 1, "unmatched": 2}))
best = best.dropna(subset=["wid"]).sort_values(["wid", "order"]).drop_duplicates("wid").set_index("wid")[gm_columns]
gm = best.reindex(master["whizdom_game_id"]).set_axis(master.index)
own = matched.set_index("game_uid")[gm_columns].reindex(master.index)
has_own = own["gm_status"].notna()
gm.loc[has_own] = own.loc[has_own]
master = master.join(gm)

# the games_enriched data of the other game_uid of the same ID only fill the empty cells
ge = [c for c in master if c.startswith("ge_")]
fill = games.dropna(subset=["whizdom_game_id"]).groupby("whizdom_game_id")[ge].first()
for column in ge:
    master[column] = master[column].fillna(master["whizdom_game_id"].map(fill[column]))
master["in_games_enriched"] = master[ge].notna().any(axis=1)
master["attributes_from"] = ml.attributes_from(master["in_games_enriched"], master["gm_status"])        # games_enriched wins
master["also_in_games_matched"] = master["in_games_enriched"] & master["gm_status"].notna()    # the same game with another game_uid
new = ["attributes_from", "in_games_enriched", "also_in_games_matched"]
columns = [c for c in master if c not in new]
after = columns.index("rounds_6m") + 1                               # next to the GGR, where the 99% list is read
master = master[columns[:after] + new + columns[after:]]
master["attributes_from"].value_counts()


In [ ]:
# url is the best link of the game, not only Slot Catalog: a game that games matched has matched can have its page on another website
# (the studio, slotslaunch, slotranker...). The order: Slot Catalog (the url of before), then the page of games matched, in this order of websites.
# A fuzzy game has only candidates that are not confirmed, so its links are not used.
LINK_ORDER = ["slotcatalog", "studio site", "studio site (attributes)", "slotslaunch", "slotranker", "slotsup", "aboutslots", "slottomat",
              "askgamblers", "slottracker", "slotjava", "slot_report", "web research"]
SITE_NAME = {"slotcatalog": "Slot Catalog", "studio site (attributes)": "Studio site", "studio site": "Studio site", "slotslaunch": "SlotsLaunch",
             "slotranker": "SlotRanker", "slotsup": "SlotsUp", "aboutslots": "AboutSlots", "slottomat": "Slottomat", "askgamblers": "AskGamblers",
             "slottracker": "SlotTracker", "slotjava": "SlotJava", "slot_report": "Slot Report", "web research": "Web research"}


def best_link(text, slotcatalog_url):                  # (url, website) of the first website in LINK_ORDER that has a link
    links = {}
    for line in str(text).split("\n"):
        found = re.match(r"^\s*(.+?):\s*(https?://\S+)\s*$", line)
        if found:
            links.setdefault(found.group(1).strip(), found.group(2))
    if isinstance(slotcatalog_url, str) and slotcatalog_url.startswith("http"):
        links["slotcatalog"] = slotcatalog_url
    for site in LINK_ORDER:
        if site in links:
            return links[site], SITE_NAME[site]
    return None, None


master["url_site"] = np.where(master["url"].notna(), "Slot Catalog", None)
fill = master["url"].isna() & master["gm_status"].eq("matched")
found = [best_link(a, b) for a, b in zip(master.loc[fill, "gm_all_links"], master.loc[fill, "gm_sc_url"])]
master.loc[fill, "url"] = [u for u, _ in found]
master.loc[fill, "url_site"] = [s for _, s in found]
master.loc[fill & master["url"].notna(), "url_source"] = "games_matched"
master["no_url"] = master["url"].isna()
after = list(master).index("url") + 1                  # url_site next to url
master = master[[c for c in master if c != "url_site"][:after] + ["url_site"] + [c for c in master if c != "url_site"][after:]]
in_99 = master["ggr_eur_6m"].notna()
print(f"url filled with another website: {int((fill & master['url'].notna()).sum())} rows | rows of the 99% list without any link: "
      f"{int((in_99 & master['url'].isna()).sum())} of {int(in_99.sum())} | {master.loc[master['url'].notna(), 'url_site'].value_counts().to_dict()}")


In [ ]:
sys.path.insert(0, str(HERE / "check_games_matched_final"))
import unify as uf  # noqa: E402     the normalizers of games matched


def volatility(text):                                  # the scale of the catalog: 'medium - high' and 'Med-High' are Medium-High
    return uf.to_volatility(str(text).replace(" - ", "-"))


def volatility_or_kept(text):                          # the clean value of games matched already is in the scale (or High-Adjustable)
    return volatility(text) or str(text).strip()


def yes_no(text):                                     # ref_games writes 1.0 and 0.0
    text = str(text).strip().lower().removesuffix(".0")
    return {"true": True, "yes": True, "1": True, "false": False, "no": False, "0": False}.get(text)


def no_zero(f):                                       # a max win of 0 is not a max win: games matched writes 0 for a table game
    return lambda text: f(text) or None


def no_na(f):                                         # Slot Catalog writes the layout N/A when it does not know
    return lambda text: None if str(f(text)).strip().upper().replace("/", "-") in ("N-A", "NA", "NONE", "") else f(text)


# texts that are an advert of the website, not a description of the game
ADVERT = re.compile(r"^(looking for an online casino|looking to play|want to play|we have scanned|if you want to start playing|don.t want to play for real money"
                    r"|you can certainly (also )?play|play .{0,150}? for free in demo mode)", re.I)
# the marks of the adverts of a website anywhere in the text: the page scraped with its offers, terms and menus, not a description
ADVERT_MARKS = re.compile(r"➤|✔️|🥇|#ad\b|T&C Apply|Gamble ?Aware|casino bonus(es)? for|bonuses for (january|february|march|april|may|june|july"
                          r"|august|september|october|november|december)|age verification", re.I)


def text_of(text):                                     # one clean text: Slot Catalog glues 'Slot' or 'Slot Review' to the start
    text = re.sub(r"\s+", " ", str(text)).strip()
    text = re.sub(r"^Slot\s*(Review)?\s*(?=[A-Z])", "", text)
    return None if text.lower() in ml.NO_TERM or ADVERT.match(text) or ADVERT_MARKS.search(text) else text


def review_of(text):                                   # 'Not found.' or only the name of the game is not a review
    text = text_of(text)
    return text if text and len(text.split()) >= 4 and text.lower() != "not found." else None


def list_text(text):                                   # pros and cons of games matched: one per |, written with ;
    return text_of("; ".join(t.strip() for t in str(text).split("|") if t.strip()))


def jackpot(text):                                     # no, yes, fixed, or the name of the jackpot (yes - jackpot cards is Jackpot Cards)
    text = re.sub(r"^yes\s*-\s*", "", str(text).strip().lower()).replace(" jackpot", "")
    if text in ml.NO_TERM or "rtp" in text:            # 'high rtp' is not a jackpot
        return None
    names = {"no": "No", "yes": "Yes", "yes fixed": "Fixed", "fixed": "Fixed", "progressive": "Progressive"}
    return names.get(text) or text.title().replace("Vip", "VIP")


def paylines(text):                                    # 20, 10 (both ways), Cluster Pays, Pay Anywhere...
    text = str(text).strip()
    if text.lower() in ml.NO_TERM:
        return None
    number = re.fullmatch(r"(\d+)\s*(\(bothway\))?", text, re.I)
    if number:
        return number.group(1) + (" (both ways)" if number.group(2) else "")
    return text.title()


def technology(text):                                  # JS, HTML5 and JS|HTML 5 are the same
    found = {t.upper().replace(" ", "") for t in re.findall(r"flash|js|html\s?5", str(text), re.I)}
    return ", ".join(sorted(found)) or None


def ref_yes(text):                                     # ref_games writes 0 also when it does not know: only its 1 is read
    return True if yes_no(text) else None


def has_words(pattern):                                # the list of features says if the game has a bonus (a vote, not the first source)
    return lambda text: bool(re.search(pattern, str(text), re.I))


def platform_date(text):                               # the platform uses 2022-01-01 as a placeholder, it is not a date
    day = uf.to_date(str(text)[:10])
    return None if day == "2022-01-01" else day


# The order of preference of the sources of each attribute (first = preferred), and why. It goes to the sheet "order of preference".
CASINO = ("games_enriched, ref_games and the Client Area are the data of the casino: they agree 100% with each other and have the value of the "
          "variant that is played (94 Low Rtp). games matched and Slot Catalog describe the base game: they agree with each other 97 to 100%, "
          "and with the casino only 80 to 88%. So the casino comes first, and the others fill what it does not have")
NOT_CASINO = ("Only Slot Catalog and games matched have it (the casino does not). Slot Catalog comes first: it is the page of the column url. "
              "games matched takes Slot Catalog first too, so the two only differ when games matched took the value from another website; "
              "games matched fills the games that Slot Catalog does not have")
# A conflict of these four is only between the sources of the casino. games matched and Slot Catalog often describe another RTP
# version of the same game (96.5 against 95.5) or rate the volatility one level apart: the value of the casino is used, it is not a conflict.
ONLY_CASINO = "Two sources of the casino (games_enriched, ref_games, Client Area) {rule}. A difference with games matched or Slot Catalog is not a conflict: they often describe another RTP version of the same game or rate it one level apart, and the value of the casino is used"
ATTRIBUTES = {
    "rtp": dict(sources=[("ge_rtp", uf.to_rtp), ("ref_rtp", uf.to_rtp), ("ca_rtp", uf.to_rtp), ("gm_rtp", uf.to_rtp), ("sc_rtp", uf.to_rtp)],
                numeric=True, tol_abs=0.15, meaning="RTP in percent", why=CASINO, conflict_columns=["ge_rtp", "ref_rtp", "ca_rtp"],
                conflict=ONLY_CASINO.format(rule="differ by more than 0.15 points")),
    "volatility": dict(sources=[("ge_volatility_type", volatility), ("ref_volatility_class", volatility), ("ca_volatility", volatility), ("gm_volatility", volatility_or_kept), ("sc_volatility", volatility)],
                       meaning="Volatility on one scale: Low, Low-Medium, Medium, Medium-High, High, Very High (also Adjusted, High-Adjustable)", why=CASINO,
                       conflict_columns=["ge_volatility_type", "ref_volatility_class", "ca_volatility"], conflict=ONLY_CASINO.format(rule="give a different level")),
    "min_bet": dict(sources=[("ge_min_bet", uf.to_number), ("ref_min_bet", uf.to_number), ("ca_min", uf.to_number), ("gm_min_bet", uf.to_number), ("sc_min_bet", uf.to_number)],
                    numeric=True, tol_abs=0.001, meaning="Minimum bet (the currency is not written in the sources)", why=CASINO,
                    conflict_columns=["ge_min_bet", "ref_min_bet", "ca_min"], conflict=ONLY_CASINO.format(rule="differ by more than 0.001")),
    "max_bet": dict(sources=[("ge_max_bet", uf.to_number), ("ref_max_bet", uf.to_number), ("ca_max", uf.to_number), ("gm_max_bet", uf.to_number), ("sc_max_bet", uf.to_number)],
                    numeric=True, tol_abs=0.001, meaning="Maximum bet (the currency is not written in the sources)", why=CASINO,
                    conflict_columns=["ge_max_bet", "ref_max_bet", "ca_max"], conflict=ONLY_CASINO.format(rule="differ by more than 0.001")),
    "max_win": dict(sources=[("sc_max_win", no_zero(uf.to_multiple)), ("gm_max_win", no_zero(uf.to_number))], numeric=True, tol_rel=0.01,
                    meaning="Max win as a multiple of the bet", why=NOT_CASINO, conflict="A difference of more than 1% between the two sources"),
    "hit_frequency": dict(sources=[("sc_hit_frequency", uf.to_number), ("gm_hit_frequency", uf.to_number)], numeric=True, tol_rel=0.01,
                          meaning="Hit frequency in percent", why=NOT_CASINO, conflict="A difference of more than 1% between the two sources"),
    "layout": dict(sources=[("sc_grid", no_na(uf.to_layout)), ("gm_layout", no_na(str))], meaning="Reels and rows (5-3, 3-4-4-4-3); a text that is not a layout is kept as written",
                   why=NOT_CASINO + ". The layout of games_enriched is not used: its column has dates (5-3 became 2026-03-05)",
                   conflict="The two sources give a different layout"),
    "release_date": dict(sources=[("sc_release_date", uf.to_date), ("gm_release_date", uf.to_date), ("ca_release_date", platform_date), ("ref_release_date", platform_date)],
                         conflict_columns=["sc_release_date", "gm_release_date"], meaning="Release date of the game (an ISO date, or a year)",
                         why=("The order is different from the other attributes on purpose. Slot Catalog and games matched give the release of the game (Slot Catalog first, as for max win). "
                              "The Client Area and ref_games give the day the game was added to the platform, which is later: they agree with Slot Catalog "
                              "in only 24% of the games, so they only fill what the others do not have. 2022-01-01 is a placeholder of the platform and is ignored"),
                         conflict="Slot Catalog and games matched give a different date (the dates of the platform are not compared)"),
    "has_buy_bonus": dict(sources=[("ge_has_buy_bonus", yes_no), ("ca_buy_bonus", yes_no), ("ref_has_buy_bonus", ref_yes), ("features", has_words(r"buy feature|bonus buy|feature buy"))],
                          conflict_columns=["ge_has_buy_bonus", "ca_buy_bonus", "ref_has_buy_bonus"], meaning="true if the game has a buy bonus",
                          why="The casino first. ref_games writes 0 also when it does not know (where it is the only source, 99.7% of its values are 0), so only its 1 (yes) is used. The list of features (Buy Feature) comes last: it agrees with the casino in 85% of the games, so it only fills the games that the casino does not know and breaks a tie. If no source knows, the value is empty",
                          conflict="Two sources of the casino say yes and no"),
    "has_freespins": dict(sources=[("ge_has_freespin", yes_no), ("ca_freespins", yes_no), ("ref_has_freespins", ref_yes), ("features", has_words(r"free ?spins"))],
                          conflict_columns=["ge_has_freespin", "ca_freespins", "ref_has_freespins"], meaning="true if the game has free spins",
                          why="The casino first. ref_games writes 0 also when it does not know (where it is the only source, 99.9% of its values are 0, but two games of three have free spins), so only its 1 (yes) is used. The list of features (Free Spins) comes last: it agrees with the casino in 82% of the games, so it only fills the games that the casino does not know and breaks a tie. If no source knows, the value is empty",
                          conflict="Two sources of the casino say yes and no"),
}
TEXT_RULE = "Not compared: the texts of two websites are always different. The first text that is a description of the game is used (an advert of the website is left out)"
ATTRIBUTES.update({
    "jackpot": dict(sources=[("ge_jackpot_type", jackpot), ("ref_jackpot", jackpot), ("ca_jackpot", jackpot)],
                    meaning="Jackpot of the game: No, Yes, Fixed, Progressive or the name of the jackpot (Jackpot Cards, Clover Chance, Bell Link...)",
                    why="Only the casino has it. games_enriched first, then ref_games and the Client Area (they agree with each other)",
                    conflict="Two sources give a different jackpot"),
    "paylines": dict(sources=[("sc_paylines", paylines), ("gm_paylines_ways", paylines)],
                     meaning="Paylines or ways to win: a number, 10 (both ways), Cluster Pays, Pay Anywhere, Scatter Pays", why=NOT_CASINO,
                     conflict="The two sources give different paylines"),
    "technology": dict(sources=[("sc_technology", technology), ("gm_technology", technology)], meaning="Technology of the game (HTML5, JS, FLASH)",
                       why=NOT_CASINO, conflict="The two sources give a different technology"),
    "description": dict(sources=[("sc_description", text_of), ("gm_description", text_of)], conflict_columns=["sc_description"], text=True,
                        meaning="Description of the game, from a third party website", why="Slot Catalog first (the page of url), then games matched", conflict=TEXT_RULE),
    "review": dict(sources=[("gm_review_text", review_of)], conflict_columns=["gm_review_text"], text=True,
                   meaning="Short review of the game, from a third party website (games matched)", why="Only games matched has it", conflict=TEXT_RULE),
    "pros": dict(sources=[("gm_pros", list_text)], conflict_columns=["gm_pros"], text=True,
                 meaning="Good points of the game, from a third party website (games matched), separated by ;", why="Only games matched has it", conflict=TEXT_RULE),
    "cons": dict(sources=[("gm_cons", list_text)], conflict_columns=["gm_cons"], text=True,
                 meaning="Weak points of the game, from a third party website (games matched), separated by ;", why="Only games matched has it", conflict=TEXT_RULE),
})
SOURCE_NAME = {"ge_": "games_enriched", "ref_": "ref_games", "ca_": "Client Area", "gm_": "games matched", "sc_": "Slot Catalog JSON", "features": "features of all the sources"}
SOLVED = "The value that most of the sources give; a tie goes to the first source in the order"


def solved(spec):                                      # a text or a single source is never compared, so nothing is solved
    return SOLVED if not spec.get("text") and len([c for c, _ in spec["sources"] if c in master]) > 1 else "-"


def order_text(spec):                                  # 1. ge_rtp (games_enriched) → 2. ref_rtp (ref_games) → ...
    used = [c for c, _ in spec["sources"] if c in master]
    return " → ".join(f"{i}. {c} ({SOURCE_NAME[c[:c.index('_') + 1] if '_' in c else c]})" for i, c in enumerate(used, 1))


# themes and features are a list, not one value: every word of every source, the same word only once (Free Spins = FreeSpins)
TERMS = {"theme": dict(columns=["ref_theme", "ca_theme", "ge_theme_y", "gm_theme", "sc_theme"], meaning="Themes of the game (Fruit, Egypt, Gods, Blue...)"),
         "features": dict(columns=["ge_features", "sc_features", "gm_features"], meaning="Features of the game (Free Spins, Wild, Multiplier, Buy Feature...)")}
for name, spec in TERMS.items():
    spec["columns"] = [c for c in spec["columns"] if c in master]
    master[name] = ml.unify_terms(master, spec["columns"])

UNIFIED = []
for name, spec in ATTRIBUTES.items():
    options = {k: v for k, v in spec.items() if k in ("numeric", "tol_abs", "tol_rel", "conflict_columns")}
    found = ml.unify_attribute(master, [(c, f) for c, f in spec["sources"] if c in master], **options)
    master[name], master[f"{name}_source"], master[f"{name}_conflict"] = found["value"], found["source"], found["conflict"]
    UNIFIED += [name, f"{name}_source", f"{name}_conflict"]
for name in ["rtp", "hit_frequency"]:                     # ref_games keeps the decimals with noise: 96.0999984741211
    master[name] = master[name].round(2)
swap = master["min_bet"] > master["max_bet"]             # some sources have them the other way round (Blackjack 26: min 2000, max 10)
master.loc[swap, ["min_bet", "max_bet"]] = master.loc[swap, ["max_bet", "min_bet"]].to_numpy()

# game type in the words of the casino: ref_games, else the Client Area, else the type of Slot Catalog or games matched
CASINO_TYPE = {"Video Slots": "Slots", "Classic Slots": "Slots", "Unassigned": None}
SC_TYPE = {"Video Slots": "Slots", "Classic Slots": "Slots", "Roulette Games": "Roulette", "Live Casino Games": "Live casino",
           "Card games": "Card games", "Scratch tickets": "Instant games", "Dice Games": "Instant games"}        # Other types says nothing
master["game_type"] = (master["game_type"].map(lambda v: CASINO_TYPE.get(v, v)).replace("", np.nan)
                       .fillna(master["sc_game_type"].map(SC_TYPE)).fillna(master["gm_game_type"].map(SC_TYPE)))
print(int(swap.sum()), "min and max bets swapped | game_type filled:", int(master["game_type"].notna().sum()))
UNIFIED += list(TERMS)
rest = [c for c in master if c not in UNIFIED]
after = rest.index("is_sports") + 1                       # the unified attributes go after the type and the sports flag
master = master[rest[:after] + UNIFIED + rest[after:]]

preference = pd.DataFrame([{"attribute": n, "meaning": s["meaning"], "order of preference (first = preferred)": order_text(s), "why this order": s["why"],
                            "sources disagree when": s["conflict"], "how it is solved": solved(s), "rows with a value": int(master[n].notna().sum()),
                            "rows where the sources disagreed (solved)": int(master[f"{n}_conflict"].sum())}
                           for n, s in ATTRIBUTES.items()]
                          + [{"attribute": n, "meaning": s["meaning"], "order of preference (first = preferred)": "no order: all of them are joined, "
                              + ", ".join(f"{c} ({SOURCE_NAME[c[:c.index('_') + 1]]})" for c in s["columns"]),
                              "why this order": "All the sources use the words of Slot Catalog, so their words are joined and each word is written once. Two words "
                                                "are the same if they only differ in case, spaces, hyphens, a final s, the end theme or the start Feature: "
                                                "(Free Spins and FreeSpins, Animals and Animal, Fruit theme and Fruit). Different words complete each other (Gold and Gold-color)",
                              "sources disagree when": "Never: the words of the sources are added, not compared", "how it is solved": "-",
                              "rows with a value": int(master[n].notna().sum()), "rows where the sources disagreed (solved)": 0} for n, s in TERMS.items()])
preference


In [ ]:
# the 99% list: how many rows (game_uid) and how much GGR each file gives (the rows with no name have no ID, so they are apart)
in_list = master[master["ggr_eur_6m"].notna()]
group = in_list["attributes_from"].where(in_list["whizdom_game_id"].notna(), "no name (no ID)")
coverage = in_list.groupby(group).agg(games=("ggr_eur_6m", "size"), ggr_eur_6m=("ggr_eur_6m", "sum"))
coverage["share_of_ggr_%"] = (coverage["ggr_eur_6m"] / coverage["ggr_eur_6m"].sum() * 100).round(1)
coverage = coverage.sort_values("ggr_eur_6m", ascending=False).reset_index(names="attributes_from")
coverage.loc[len(coverage)] = ["total", coverage["games"].sum(), coverage["ggr_eur_6m"].sum(), 100.0]

# the games of games matched with no match yet: the work that is left, most GGR first (game_uid is the key of the file)
pending = matched[matched["gm_status"] == "unmatched"][["game_uid", "gm_batch", "gm_unmatched_reason"]]
pending = pending.merge(games[["ref_game_uid", "whizdom_game_id", "game_name", "studio", "ggr_eur_6m", "is_sports"]], left_on="game_uid", right_on="ref_game_uid", how="left").drop(columns="ref_game_uid")
pending["state"] = np.where(pending["is_sports"].eq(True), "sports (no page)", "pending")      # sports betting has no Slot Catalog page
pending = pending.sort_values("ggr_eur_6m", ascending=False)[["state", "game_uid", "whizdom_game_id", "game_name", "studio", "is_sports", "ggr_eur_6m",
                                                              "gm_batch", "gm_unmatched_reason"]].reset_index(drop=True)
print(pending["state"].value_counts().to_dict())
coverage


## 10. Output

- `master_catalog.xlsx`: the sheet `read me` says what each sheet is and what to expect in it.
- `master_games.parquet`: the same catalog.
- `game_identities.parquet`: each `game_uid` and its ID, to join the activity.
- `raw_sources.parquet`: the full raw JSON.


In [ ]:
raw_sources = raw.merge(js[["sc_uuid", "whizdom_game_id"]], left_on="source_id", right_on="sc_uuid", how="left").drop(columns="sc_uuid")
raw_sources.insert(0, "source", "slotcatalog_json")

# the parquet keeps the game_uid of each row (empty for the games only in other sources), to join other files; the Excel does not
master.assign(game_uid=master.index.where(master["only_in"].fillna("").eq(""))).reset_index(drop=True).to_parquet(OUT / "master_games.parquet", index=False)
identities.to_parquet(OUT / "game_identities.parquet", index=False)
raw_sources.to_parquet(OUT / "raw_sources.parquet", index=False)

In [ ]:
FLAG_MEANING = {"rows of the catalog": "Rows of the sheet catalog: one per game_uid of ref_games, plus the games only in other sources",
                "exact": "match_status exact: the Slot Catalog page found by the platform ids is the same game",
                "review": "match_status review: the link with Slot Catalog is not sure (see the sheet review)",
                "no_match": "match_status no_match: no Slot Catalog page was found",
                "no url": "Rows with no Slot Catalog page: url is empty"}
notes = {**{c: {"meaning": v["meaning"]} for c, v in yaml.safe_load((HERE / "columns.yaml").read_text()).items()},
         **yaml.safe_load((HERE / "master_columns.yaml").read_text())}
# the warnings of the columns of the catalog; the disagreements of the sources are solved and counted in the sheet order of preference
flag_columns = ["missing_data", "similar_provider", "id_shared", "same_platform_conflict"]
counts = {"rows of the catalog": len(master), **{s: master["match_status"].eq(s).sum() for s in ("exact", "review", "no_match")},
          "no url": master["url"].isna().sum(),
          **{c: (master[c].ne("") if c == "missing_data" else master[c]).sum() for c in flag_columns}}
flags = pd.DataFrame({"flag": list(counts), "rows": [int(v) for v in counts.values()],
                      "meaning": [FLAG_MEANING.get(f) or notes[f]["meaning"] for f in counts]})

# the other provider with almost the same name (the same core without gaming, casino, live...), so a person can decide
cores = candidates[["core", "standard"]].drop_duplicates()
pairs = cores.merge(cores, on="core").query("standard_x != standard_y")
similar_to = pairs.groupby("standard_x")["standard_y"].agg(lambda s: ", ".join(sorted(set(s))))
providers = games.dropna(subset=["provider_id"]).groupby(["provider_id", "provider_slug"]).agg(
    names=("studio", lambda s: ", ".join(sorted(set(s.dropna())))), games=("whizdom_game_id", "nunique")).reset_index().rename(columns={"provider_slug": "provider"})
providers["similar_to"] = providers["provider"].map(similar_to).fillna("")
providers = providers.sort_values("games", ascending=False)

review = games[games["match_status"].eq("review")][["whizdom_game_id", "game_name", "studio", "provider_slug", "ref_provider_code", "close_name", "url"]]
no_name = games[games["whizdom_game_id"].isna()][["ref_provider_code", "ref_provider_name", "ref_game_id", "studio", "game_type", "url", "ggr_eur_6m", "rounds_6m"]]
no_name = no_name.sort_values(["ggr_eur_6m", "rounds_6m"], ascending=False, na_position="last")      # most GGR first

for n in GM_FIELDS:                                   # the unified fields of games matched are explained by the rules of their file
    spec = gm_fields[n]
    origin = ", ".join(([spec["attr"]] if spec.get("attr") else []) + spec.get("sources", []) + (["crawler_records." + spec["records"]] if spec.get("records") else []))
    gm = {"source": "games matched, one clean value per field", "origin": origin}
    notes[f"gm_{n}"] = {**gm, "meaning": f"{n}, one clean value from games matched", "how": f"First value found in: {spec['order']}. {spec['normalize']}"}
    notes[f"gm_{n}_source"] = {**gm, "meaning": f"Where gm_{n} comes from", "how": f"The column or website of games matched that gave gm_{n}."}
for n, spec in ATTRIBUTES.items():                    # the unified attributes are explained by their order of preference
    origin = ", ".join(c for c, _ in spec["sources"] if c in master)
    several = len([c for c, _ in spec["sources"] if c in master]) > 1
    notes[n] = {"source": "unified (sheet order of preference)", "origin": origin, "meaning": spec["meaning"] + (". One value out of several sources" if several else ""),
                "how": f"The first value found in this order: {order_text(spec)}. Why: {spec['why']}"
                       + (f". When the sources disagree: {SOLVED.lower()}" if solved(spec) != "-" else "")}
    notes[f"{n}_source"] = {"source": "unified (sheet order of preference)", "origin": origin, "meaning": f"The column that gave {n}", "how": f"The first of the sources of {n} that has a value."}
    notes[f"{n}_conflict"] = {"source": "unified (sheet order of preference)", "origin": origin, "meaning": f"true if the sources of {n} disagreed; {n} has the value that most of them give",
                              "how": f"The normalized values of the sources are compared. They disagree when: {spec['conflict']}. Solved: {SOLVED.lower()}."}
originals = {**{"ref_" + c: c for c in ref.columns}, **{"ca_" + ml.snake(c): c for c in ca.columns}, **{"scu_" + c: c for c in sa.columns},
             **{"ge_" + c: c for c in enriched.columns}, **{v: "raw_data." + k for k, v in slotcatalog_json.FIELDS.items()}}
sources = ml.source_notes(HERE / "master_sources.yaml", HERE, {
    "ref_": len(ref), "ggr_eur_6m, rounds_6m": len(activity), "ca_": len(ca), "scu_": len(sa), "ge_": len(enriched), "sc_": len(raw),
    "gm_": len(matched), "names written by hand": len(config["explicit_names"]), "provider list": len(set(aliases.values()))})

# the final catalog: the columns to read; master_catalog_full.xlsx has every column. The platform is the one of the row
master["first_seen"], master["last_seen"] = (master[c].astype(str).str[:10].where(master[c].notna()) for c in ["ref_first_seen", "ref_last_seen"])
master["vertical"] = master["ref_vertical"]
master["platform"] = master["ref_provider_code"].fillna(master.get("ca_platform")).fillna(master.get("scu_platform")).fillna(master.get("ge_platform_code"))
TEXTS = [n for n, s in ATTRIBUTES.items() if s.get("text")]          # long texts go at the end, so the catalog reads well
FINAL = (["whizdom_game_id", "game_name", "ggr_eur_6m", "rounds_6m", "first_seen", "last_seen", "attributes_from", "provider", "studio", "status",
          "vertical", "game_type", "is_sports"]
         + [n for n in ATTRIBUTES if n not in TEXTS] + list(TERMS)   # one fixed value each: the conflicts are solved (_source and _conflict in the full file)
         + ["url", "url_site", "match_status", "missing_data", "similar_provider", "id_shared", "same_platform_conflict"] + TEXTS)
assert all(c in master for c in FINAL), [c for c in FINAL if c not in master]
n_ggr, n_master = int(with_ggr.sum()), len(master)
SHEETS = {
    "read me": "This sheet: what each sheet is, the numbers of this version and the rules of the catalog.",
    "catalog": f"One row per game_uid of ref_games (the list of ClickHouse), plus the games that are only in other sources (the last rows): {n_master} rows, "
               "the same rows as the Games Catalog Tracker. A game on several platforms has one Whizdom Game ID in several rows. "
               f"The first {n_ggr} rows are the 99% list (GGR of the last 6 months), highest GGR first; the rows with no GGR go after. The first columns are the ID, the name, the GGR, the first and last day the game "
               "was seen, where the attributes come from (attributes_from), the provider, the status, the vertical and the type. Then the unified attributes "
               "(rtp, volatility, bets, max win, hit frequency, layout, release date, buy bonus, free spins, jackpot, paylines, technology), one fixed value "
               "each (when the sources disagree, the value that most of them give), and the themes and features of all the sources in one column each. "
               "url opens the page of the game: Slot Catalog if it has one, else another website (url_site says which). Then the checks of the data, and at the end the texts of third party websites (description, review, pros, cons). "
               f"The final columns only ({len(FINAL)}): master_catalog_full.xlsx has the same rows with every column of every source.",
    "columns": "One row per column of the sheet catalog, in the same order: its source, the columns it comes from, how it is made, its meaning, the "
               "percent of rows with a value and one example. Read it to understand any column of the catalog. The columns that it names and that "
               "are not in the catalog (the columns of each source, with ref_, ca_, scu_, sc_, ge_ or gm_, and the _source and _conflict columns) "
               "are in master_catalog_full.xlsx.",
    "order of preference": "One row per unified attribute: the sources in order of preference (the first one that has a value wins), why that order, "
                           "when the sources disagree, how it is solved, how many rows have a value and in how many rows the sources disagreed.",
    "sources": "One row per source of data: the table or file, how it is read, how it joins to the catalog, the query (for ClickHouse) and its rows.",
    "coverage 99%": "The rows of the 99% list grouped by the file that gives their attributes (games_enriched, games matched, or neither), with their GGR "
                    "and share of GGR. The total is 100% of the GGR of the list.",
    "pending unmatched": "The games of games matched that have no Slot Catalog match yet, highest GGR first. state is pending, or sports (no page) "
                         "for sports betting, which has no Slot Catalog page. It is the work that is left.",
    "providers": "One row per Provider ID: the standard provider, the studio names that give it, its number of games, and similar_to: another "
                 "provider with almost the same name, which can be the same company (a person decides if they are merged).",
    "flags": "How many rows of the catalog have each warning (match with Slot Catalog, no URL, missing data, similar provider, shared ID) and what it means. "
             "The rows where the sources disagreed are not a warning: they are solved, and the sheet order of preference counts them.",
    "review": "The game_uid whose link with Slot Catalog is not sure (match_status review), with a close name and the URL, to check by hand.",
    "no name": "The game_uid that have no name in any source, so they have no Whizdom Game ID (in the catalog their ID is empty), with their platform id. Highest GGR first.",
}
gm_date = pd.Timestamp((HERE / config["navid_final"]).stat().st_mtime, unit="s").date()
facts = {
    "built on": str(pd.Timestamp.today().date()),
    "ref_games read from ClickHouse on": str(REF_DATE),
    "id version": ml.ID_VERSION,
    "games of ref_games": len(ref),
    "games with no name (no Whizdom Game ID, sheet no name)": len(no_name),
    "names taken from the Client Area, the Slot Catalog URL file or games_enriched": int(games["name_source"].isin(["client_area", "slotcatalog_file", "games_enriched"]).sum()),
    "names written by hand (sports betting games with no name in any source)": int(games["name_source"].eq("explicit").sum()),
    "studios taken from the same game name on another platform": int(games["studio_source"].eq("same_name").sum()),
    "columns left out because they are empty in every row": len(empty),
    "columns of the catalog (final)": len(FINAL),
    "columns of master_catalog_full.xlsx (every source)": master.shape[1],
    "rows of the catalog": n_master,
    "rows of ref_games (one per game_uid)": len(ref),
    "rows of games only in other sources (the last rows)": len(others),
    "different Whizdom Game IDs": int(master["whizdom_game_id"].nunique()),
    "Provider IDs (sheet providers)": len(providers),
    "Whizdom Game IDs changed against the last version": int(changes["change"].eq("changed").sum()),
    "GGR and rounds": f"last 6 full months, the 99% list of {ACTIVITY_DATE} (games that hold 99% of the positive GGR)",
    "rows with GGR (the first rows, the 99% list)": n_ggr,
    "GGR of the rows with no name, so no Whizdom Game ID (EUR)": round(float(games.loc[games["whizdom_game_id"].isna(), "ggr_eur_6m"].sum())),
    "games matched": f"games_matched_final.xlsx of {gm_date}",
    "rows with data of games matched": int(master["gm_status"].notna().sum()),
    "rows with data of games_enriched": int(master["in_games_enriched"].sum()),
}
RULES = {
    "column prefixes": "ref_ ref_games, ca_ Client Area export, scu_ Slot Catalog URL file, sc_ Slot Catalog JSON, ge_ games_enriched, gm_ games matched (one clean value per field); no prefix: built for the catalog",
    "order of the rows": "GGR (highest first), rounds (highest first), game_uid, as the 99% list; the first rows are the 99% list. Then the rows with no GGR; at the end the games only in other sources",
    "one row per game_uid": "the rows are the list of ref_games, one per game on each platform; a game on several platforms has one Whizdom Game ID in several rows (master_catalog_full.xlsx has the platform of each row)",
    "unified attributes": "one fixed value per row out of several sources, in the order of the sheet order of preference; when the sources disagree, the value that most of them give (a tie goes to the first in the order). master_catalog_full.xlsx has the value of each source, _source (which one gave the value) and _conflict (the sources disagreed)",
    "ids of platforms": "the ids of platforms and brands are not in the catalog: the Whizdom Game ID does not depend on them",
}
read_me = pd.DataFrame([("sheet", k, v) for k, v in SHEETS.items()] + [("numbers", k, v) for k, v in facts.items()]
                       + [("rules", k, v) for k, v in RULES.items()], columns=["topic", "item", "value"])


In [ ]:
sheets = {"read me": read_me, "catalog": master[FINAL], "columns": ml.column_notes(master[FINAL], notes, originals), "order of preference": preference,
          "sources": sources, "coverage 99%": coverage, "pending unmatched": pending, "providers": providers, "flags": flags, "review": review, "no name": no_name}
assert list(sheets) == list(SHEETS), "the read me must explain every sheet, in the same order"
hits = ml.forbidden_cells({k: v for k, v in sheets.items()})
assert not hits, f"{len(hits)} cells name a person or a local file, for example {hits[:5]}"
ml.write_excel(OUT / "master_catalog.xlsx", sheets, freeze={"catalog": "D2"}, links={"catalog": "url", "review": "url", "no name": "url"},
               wide={"read me": {"item": 45, "value": 120}, "columns": {"source": 30, "origin": 40, "how": 90, "meaning": 55, "example": 30},
                     "order of preference": {"meaning": 45, "order of preference (first = preferred)": 70, "why this order": 90, "sources disagree when": 50, "how it is solved": 50},
                     "sources": {"source": 30, "where": 40, "how": 70, "joined": 60, "query": 90}, "flags": {"meaning": 90}})

# the same rows with every column of every source, for whoever needs the detail
full_read_me = pd.DataFrame([("file", "what it is", "Every column of every source for each row of master_catalog.xlsx: the same rows, in the same order. "
                                                     "master_catalog.xlsx has the final columns; its sheet read me explains the rest"),
                             ("file", "columns", f"{master.shape[1]} (the final catalog has {len(FINAL)})"),
                             ("file", "built on", str(pd.Timestamp.today().date()))], columns=["topic", "item", "value"])
full = {"read me": full_read_me, "catalog": master, "columns": ml.column_notes(master, notes, originals)}
hits = ml.forbidden_cells(full)
assert not hits, f"{len(hits)} cells name a person or a local file, for example {hits[:5]}"
ml.write_excel(OUT / "master_catalog_full.xlsx", full, freeze={"catalog": "D2"}, links={"catalog": "url"},
               wide={"read me": {"item": 30, "value": 120}, "columns": {"source": 30, "origin": 40, "how": 90, "meaning": 55, "example": 30}})
print(f"catalog {len(master)} rows x {len(FINAL)} columns (full: {master.shape[1]}) | providers {len(providers)} | no name {len(no_name)} -> {OUT}")
flags


## 11. Games Catalog Tracker

A separate Excel with the list of ref_games as it is (one row per game_uid), completed with the Whizdom Game ID and four states. At the end, the games that are only in other sources (no game_uid).
- **Raw Data**: Yes if there is raw page data (Slot Catalog JSON of the game_uid, or games matched matched or fuzzy).
- **Mapping**: Mapped (Slot Catalog exact, games_enriched or games matched matched), To review (Slot Catalog review or games matched fuzzy), else Not mapped.
- **Status**: the commercial status and ready to use (released, Mapped, RTP and volatility without conflict), needs review or not ready. Notes says why.
- **Scope**: In scope if the game_uid is in the 99% GGR list.


In [ ]:
# the tracker is the catalog with the columns that the boss asked for: the same rows, in the same order
id_note = pd.Series("platform " + games["ref_provider_code"].astype(str) + ", game id " + games["ref_game_id"].astype(str)).set_axis(games["ref_game_uid"])
only = master["only_in"].fillna("")
rows = master.assign(id_note=master.index.map(id_note),
                     source_note=np.where(only.ne(""), "only in " + only + ", not in ref_games" + np.where(master["whizdom_game_id"].isna(), ", no provider in any source", ""), ""))
tracker = ml.tracker(rows)
assert len(tracker) == len(master), "the tracker has the rows of the catalog"

COLUMNS = {
    "whizdom_game_id": "Whizdom Game ID, the same as whizdom_game_id of master_catalog.xlsx: uuid5 of the provider and the game name, the same for ever. The rows are "
               "the list of ref_games, one per game on each platform, so a game on several platforms has one Game ID in several rows. Empty if the game has no name or no provider in any source",
    "Game Name": "Name of the game (ref_games, else the Client Area, the Slot Catalog URL file or games_enriched)",
    "Game Provider": "Standard provider (the studio that made the game, not the platform), one per Provider ID",
    "External URL": "Page of the game (click to open): Slot Catalog, else the page that games matched found on another website (the studio, SlotsLaunch, SlotRanker...). The same as url in master_catalog.xlsx. Empty if none; the links of a fuzzy match are candidates and are not used",
    "Raw Data": "Yes if there is raw page data for the game: a Slot Catalog JSON linked to this game_uid, or the data of games matched "
                "(matched or fuzzy). No if not",
    "Mapping": "Mapped: the raw data is linked to the game for sure (Slot Catalog exact, games_enriched, or games matched matched). "
               "To review: there is a candidate, not confirmed (Slot Catalog review or games matched fuzzy). Not mapped: no link",
    "Status": "The commercial status of this game_uid (released, disabled, review, upcoming, unknown) and if the game is ready to use. "
              "ready to use: released, Mapped, with an RTP and a volatility (one fixed value each). needs review: released "
              "and Mapped or To review, but a check fails. not ready: no name, not released or not mapped",
    "Scope": "In scope: the game_uid is in the 99% list (the games that hold 99% of the positive GGR of the last 6 months, list of 2026-10-28), "
             "the list of the current review. Out of scope: the rest",
    "Notes": "Why the game is not ready to use (empty if it is ready)",
}
assert list(COLUMNS) == list(tracker.columns), "every column of the tracker must be explained"
counts = {c: tracker[c].value_counts().to_dict() for c in ["Raw Data", "Mapping", "Status"]}
in_scope = tracker["Scope"].eq("In scope")
tracker_read_me = pd.DataFrame(
    [("column", c, t) for c, t in COLUMNS.items()]
    + [("numbers", "built on", str(pd.Timestamp.today().date())), ("numbers", "ref_games read from ClickHouse on", str(REF_DATE)),
       ("numbers", "rows", len(tracker)), ("numbers", "rows of ref_games (one per game_uid)", len(ref)),
       ("numbers", "rows of games only in other sources (no game_uid)", len(others)), ("numbers", "different Game IDs", int(tracker["whizdom_game_id"].nunique())),
       ("numbers", "rows with no Game ID (no name, or no provider in any source)", int(tracker["whizdom_game_id"].isna().sum())),
       ("numbers", "rows in scope", int(in_scope.sum())), ("numbers", "rows out of scope", int((~in_scope).sum()))]
    + [("numbers", f"{c}: {v}", f"{n} rows, {int((tracker[c].eq(v) & in_scope).sum())} in scope") for c, d in counts.items() for v, n in d.items()]
    + [("rules", "order of the rows", "GGR of the last 6 months (highest first), as the 99% list, so the rows in scope come first; then the rest of ref_games; at the end the games only in other sources"),
       ("rules", "by game_uid or by game", "URL, Raw Data (Slot Catalog JSON), Slot Catalog match, status and scope are of the game_uid; "
                                           "games_enriched, games matched, RTP and volatility are of the game (Game ID) on any platform"),
       ("rules", "games only in other sources", "the last rows: the games of the Client Area, games_enriched, the Slot Catalog URL file or the Slot Catalog JSON "
                                                "that are not in ref_games and are not already in the list with another key. Notes says the source. They have no GGR, so they are out of scope"),
       ("rules", "more detail", "master_catalog.xlsx has the final columns of each row, and master_catalog_full.xlsx every column of every source")],
    columns=["topic", "item", "value"])

# key numbers at a glance: each one for all the rows, in scope and out of scope
ready = tracker["Status"].str.endswith("ready to use")
raw_yes = tracker["Raw Data"].eq("Yes")
METRICS = [
    ("all games", pd.Series(True, index=tracker.index), "Every row of the tracker: the game_uid of ref_games and the games only in other sources"),
    ("ready to use", ready, "Status ends with ready to use: released, Mapped, with an RTP and a volatility"),
    ("raw data available", raw_yes, "Raw Data = Yes: a Slot Catalog JSON or the data of games matched"),
    ("mapped", tracker["Mapping"].eq("Mapped"), "Mapping = Mapped: the raw data is linked to the game for sure"),
    ("to do: extract the data", ~ready & ~raw_yes, "Not ready to use and Raw Data = No: the data of the game must be extracted first"),
    ("to do: touchup or full mapping", ~ready & raw_yes, "Not ready to use but Raw Data = Yes: the data is there, the mapping or a missing value must be fixed "
                                                         "(see Mapping and Notes)"),
]
summary = pd.DataFrame([{"metric": name, "all": int(m.sum()), "in scope": int((m & in_scope).sum()), "out of scope": int((m & ~in_scope).sum()),
                         "% of all": round(100 * m.mean(), 1), "% of in scope": round(100 * (m & in_scope).sum() / in_scope.sum(), 1), "how it is counted": how}
                        for name, m, how in METRICS])
assert summary.loc[summary["metric"].isin(["ready to use", "to do: extract the data", "to do: touchup or full mapping"]), "all"].sum() == len(tracker), \
    "ready + extract + touchup must be every row"
tracker_read_me.loc[len(tracker_read_me)] = ["rules", "summary", "The sheet summary has the key numbers: ready to use, in scope and out of scope, "
                                             "raw data, mapped, and the work left (extract the data, or touchup and mapping). Ready + extract + touchup = all"]

tracker_sheets = {"summary": summary, "tracker": tracker, "read me": tracker_read_me}
hits = ml.forbidden_cells(tracker_sheets)
assert not hits, f"{len(hits)} cells name a person or a local file, for example {hits[:5]}"
ml.write_excel(OUT / "Games Catalog Tracker.xlsx", tracker_sheets, freeze={"tracker": "C2"}, links={"tracker": "External URL"},
               wide={"summary": {"metric": 32, "how it is counted": 90}, "tracker": {"Game Name": 40, "Notes": 60}, "read me": {"item": 45, "value": 120}})
print(len(tracker), "rows ->", OUT / "Games Catalog Tracker.xlsx")
summary
